In [9]:
import os
import math
import optuna
import numpy as np
import pandas as pd
import warnings

In [10]:
warnings.simplefilter("ignore")

In [11]:
path="./data"

if not os.path.exists(path):
  os.mkdir(path)

dp = f"{path}/thesis-data.csv"

# Load or Download Data
if os.path.exists(dp):
    data = pd.read_csv(dp, low_memory=False)
# Preprocess Data
if 'date' not in data.columns:
    data.reset_index(inplace=True)  # Reset index to bring 'date' as a column

# Convert column names to lowercase
data.columns = data.columns.str.lower()

# Ensure 'date' is the index
data["date"] = pd.to_datetime(data["date"])  # Convert to datetime format
data.set_index("date", inplace=True)
# data = data.drop("unnamed: 0", axis=1)

In [12]:
data = data[['open', 'high', 'low', 'close', 'volume']]

In [13]:
data

,open,high,low,close,volume
date,,,,,
1965-01-05,1257.719971,1257.719971,1257.719971,1257.719971,0
1965-01-06,1263.989990,1263.989990,1263.989990,1263.989990,0
1965-01-07,1274.270020,1274.270020,1274.270020,1274.270020,0
1965-01-08,1286.430054,1286.430054,1286.430054,1286.430054,0
1965-01-12,1288.540039,1288.540039,1288.540039,1288.540039,0
...,...,...,...,...,...
2025-02-20,4695.439941,4764.029785,4675.330078,4761.939941,0
2025-02-21,4766.169922,4790.049805,4755.020020,4775.060059,0
2025-02-24,4774.359863,4774.589844,4678.839844,4678.839844,0


In [29]:
# --------------------------------------------------------------------------
# Helper Functions (very simplified placeholders for demonstration)
# --------------------------------------------------------------------------
def calculate_average(values):
    """Simple helper to compute average (mean)."""
    return np.mean(values)

def calculate_standard_deviation(values):
    """Simple helper to compute standard deviation."""
    return np.std(values, ddof=1)  # ddof=1 for sample std

def calculate_stochastic_rsi(values, period=14):
    """
    Placeholder Stochastic RSI calculation.
    Returns (k, d). 
    Replace with your actual Stochastic RSI logic as needed.
    """
    if len(values) < period:
        return (np.nan, np.nan)
    min_val = np.min(values[-period:])
    max_val = np.max(values[-period:])
    if max_val - min_val == 0:
        k_value = 50.0
    else:
        k_value = (values[-1] - min_val) / (max_val - min_val) * 100
    d_value = k_value  # For demo, just set d = k
    return (k_value, d_value)

# --------------------------------------------------------------------------
# Main Class
# --------------------------------------------------------------------------
class StockIndicators:
    def __init__(self, df):
        """
        df must have columns: 'open', 'high', 'low', 'close', 'volume'
        """
        self.df = df.copy()
        self.opens = df['open'].values
        self.highs = df['high'].values
        self.lows = df['low'].values
        self.closes = df['close'].values
        self.volumes = df['volume'].values
        self.original_columns = set(self.df.columns)

    # ----------------------------------------------------------------
    # 1) Gaussian Channel
    # ----------------------------------------------------------------
    def gaussian_channel(self, period=20):
        """
        Calculates rolling mean, std, and upper_band for the closes.
        Also calculates and stores Stoch RSI (k, d) for reference.
        """
        mean_list = []
        std_list = []
        upper_list = []
        k_list = []
        d_list = []

        for i in range(len(self.closes)):
            if i < period:
                mean_list.append(np.nan)
                std_list.append(np.nan)
                upper_list.append(np.nan)
            else:
                window = self.closes[i-period:i]
                mean_val = calculate_average(window)
                std_val = calculate_standard_deviation(window)
                upper_band = mean_val + 2 * std_val
                mean_list.append(mean_val)
                std_list.append(std_val)
                upper_list.append(upper_band)

            # Also store Stochastic RSI at each bar
            k_val, d_val = calculate_stochastic_rsi(self.closes[:i+1])
            k_list.append(k_val)
            d_list.append(d_val)

        self.df['gaussian_mean'] = mean_list
        self.df['gaussian_std'] = std_list
        self.df['gaussian_upper'] = upper_list
        self.df['stoch_k'] = k_list
        self.df['stoch_d'] = d_list

    # ----------------------------------------------------------------
    # 2) MA Crossover
    # ----------------------------------------------------------------
    def ma_crossover(self, fast_period=10, slow_period=20):
        """
        Stores a rolling fast MA and slow MA.
        """
        fast_list = []
        slow_list = []

        for i in range(len(self.closes)):
            if i < slow_period:
                fast_list.append(np.nan)
                slow_list.append(np.nan)
            else:
                fast_ma = calculate_average(self.closes[i - fast_period : i])
                slow_ma = calculate_average(self.closes[i - slow_period : i])
                fast_list.append(fast_ma)
                slow_list.append(slow_ma)

        self.df['ma_fast'] = fast_list
        self.df['ma_slow'] = slow_list

    # ----------------------------------------------------------------
    # 3) Bollinger Bands
    # ----------------------------------------------------------------
    def bollinger_bands(self, period=20):
        """
        Stores Bollinger mean, std, upper band, lower band.
        """
        means = []
        stds = []
        ups = []
        downs = []

        for i in range(len(self.closes)):
            if i < period:
                means.append(np.nan)
                stds.append(np.nan)
                ups.append(np.nan)
                downs.append(np.nan)
            else:
                window = self.closes[i - period : i]
                mean_val = calculate_average(window)
                std_val = calculate_standard_deviation(window)
                up = mean_val + (2 * std_val)
                down = mean_val - (2 * std_val)
                means.append(mean_val)
                stds.append(std_val)
                ups.append(up)
                downs.append(down)

        self.df['boll_mean'] = means
        self.df['boll_std'] = stds
        self.df['boll_up'] = ups
        self.df['boll_down'] = downs

    # ----------------------------------------------------------------
    # 4) Parabolic SAR (very simplified, per snippet)
    # ----------------------------------------------------------------
    def parabolic_sar(self):
        """
        Example from snippet:
          parabolic SAR = min of all lows up to that point (if i>=2), else NaN.
        This is NOT the standard SAR formula, just a demonstration to match snippet.
        """
        sar_list = []
        for i in range(len(self.lows)):
            if i < 2:
                sar_list.append(np.nan)
            else:
                # the snippet does: sar = min(self.lows[:i])
                # We'll store that for each row
                sar_val = np.min(self.lows[:i])
                sar_list.append(sar_val)

        self.df['parabolic_sar'] = sar_list

    # ----------------------------------------------------------------
    # 5) Ichimoku Cloud (tenkan, kijun, senkou_a)
    # ----------------------------------------------------------------
    def ichimoku_cloud(self, period=26):
        tenkan_list = []
        kijun_list = []
        senkou_a_list = []

        for i in range(len(self.closes)):
            if i < period:
                tenkan_list.append(np.nan)
                kijun_list.append(np.nan)
                senkou_a_list.append(np.nan)
            else:
                # tenkan uses 9 bars
                tenkan_val = (np.max(self.highs[i-9:i]) + np.min(self.lows[i-9:i])) / 2
                kijun_val = (np.max(self.highs[i-period:i]) + np.min(self.lows[i-period:i])) / 2
                senkou_a_val = (tenkan_val + kijun_val) / 2
                tenkan_list.append(tenkan_val)
                kijun_list.append(kijun_val)
                senkou_a_list.append(senkou_a_val)

        self.df['ichimoku_tenkan'] = tenkan_list
        self.df['ichimoku_kijun'] = kijun_list
        self.df['ichimoku_senkou_a'] = senkou_a_list

    # ----------------------------------------------------------------
    # 6) MACD
    # ----------------------------------------------------------------
    def macd(self, fast=12, slow=26, signal=9):
        """
        We'll store macd_line and macd_signal as rolling averages 
        (again, simplified approach).
        """
        macd_line_list = []
        signal_line_list = []

        for i in range(len(self.closes)):
            if i < slow:
                macd_line_list.append(np.nan)
                signal_line_list.append(np.nan)
            else:
                ema_fast = calculate_average(self.closes[i - fast : i])
                ema_slow = calculate_average(self.closes[i - slow : i])
                macd_val = ema_fast - ema_slow
                
                # For the signal line, we'll just do a (signal)-period average of closes 
                # per the snippet logic
                if i < signal:
                    sig_val = np.nan
                else:
                    sig_val = calculate_average(self.closes[i - signal : i])
                
                macd_line_list.append(macd_val)
                signal_line_list.append(sig_val)

        self.df['macd_line'] = macd_line_list
        self.df['macd_signal'] = signal_line_list

    # ----------------------------------------------------------------
    # 7) RSI (based on snippet: using stoch RSI again)
    # ----------------------------------------------------------------
    def rsi(self, period=14):
        """
        We'll store a 'rsi_value' column, using the snippet's approach 
        (which actually uses the K from Stochastic RSI).
        """
        rsi_vals = []
        for i in range(len(self.closes)):
            # Calculate stoch RSI
            k_val, _ = calculate_stochastic_rsi(self.closes[:i+1], period=period)
            rsi_vals.append(k_val)

        self.df['rsi_value'] = rsi_vals

    # ----------------------------------------------------------------
    # 8) ADX (based on snippet approach)
    # ----------------------------------------------------------------
    def adx(self, period=14):
        """
        Very simplified snippet-based ADX:
          up_moves = [high[j] - high[j-1]]
          down_moves = [low[j-1] - low[j]]
          adx = average of abs(up_moves - down_moves)
        for the last 'period' bars.
        """
        adx_list = []
        for i in range(len(self.closes)):
            if i < period:
                adx_list.append(np.nan)
            else:
                up_moves = []
                down_moves = []
                for j in range(i-period+1, i+1):
                    up_moves.append(self.highs[j] - self.highs[j-1])
                    down_moves.append(self.lows[j-1] - self.lows[j])
                # Pairwise difference
                diffs = [abs(u - d) for u, d in zip(up_moves, down_moves)]
                adx_val = calculate_average(diffs)
                adx_list.append(adx_val)

        self.df['adx_value'] = adx_list

    # ----------------------------------------------------------------
    # 9) ATR
    # ----------------------------------------------------------------
    def atr(self, period=14):
        """
        ATR per snippet:
          tr = max(high-low, abs(high-prev_close), abs(low-prev_close))
          atr = average(tr over period).
        """
        atr_list = []
        for i in range(len(self.closes)):
            if i < period:
                atr_list.append(np.nan)
            else:
                tr_values = []
                for j in range(i-period+1, i+1):
                    high_low = self.highs[j] - self.lows[j]
                    high_close = abs(self.highs[j] - self.closes[j-1])
                    low_close = abs(self.lows[j] - self.closes[j-1])
                    tr_values.append(max(high_low, high_close, low_close))
                atr_val = calculate_average(tr_values)
                atr_list.append(atr_val)

        self.df['atr_value'] = atr_list

    # ----------------------------------------------------------------
    # 10) VWMA
    # ----------------------------------------------------------------
    def vwma(self, period=20):
        """
        Volume Weighted Moving Average:
          price_vol = sum(closes * volume)
          vol_sum = sum(volume)
          vwma = price_vol / vol_sum
        """
        vwma_list = []
        for i in range(len(self.closes)):
            if i < period:
                vwma_list.append(np.nan)
            else:
                window_close = self.closes[i - period : i]
                window_vol = self.volumes[i - period : i]
                price_vol = np.sum(window_close * window_vol)
                vol_sum = np.sum(window_vol)
                if vol_sum == 0:
                    vwma_val = np.nan
                else:
                    vwma_val = price_vol / vol_sum
                vwma_list.append(vwma_val)

        self.df['vwma_value'] = vwma_list

    # ----------------------------------------------------------------
    # 11) Donchian Channel
    # ----------------------------------------------------------------
    def donchian_channel(self, period=20):
        """
        We'll just store the upper channel from the snippet.
        """
        upper_list = []
        for i in range(len(self.closes)):
            if i < period:
                upper_list.append(np.nan)
            else:
                upper_channel = np.max(self.highs[i - period : i])
                upper_list.append(upper_channel)

        self.df['donchian_upper'] = upper_list

    # ----------------------------------------------------------------
    # 12) Keltner Channel
    # ----------------------------------------------------------------
    def keltner_channel(self, period=20):
        """
        Snippet version:
          typical_price = (h + l + c)/3
          ema = average(typical_price)
          atr = average([h-l])
          upper_band = ema + 2*atr
        We'll store 'keltner_ema', 'keltner_atr', 'keltner_upper'.
        """
        ema_list = []
        atr_list = []
        upper_list = []

        for i in range(len(self.closes)):
            if i < period:
                ema_list.append(np.nan)
                atr_list.append(np.nan)
                upper_list.append(np.nan)
            else:
                tp_window = []
                hl_diff = []
                for j in range(i - period, i):
                    tp_window.append((self.highs[j] + self.lows[j] + self.closes[j]) / 3)
                    hl_diff.append(self.highs[j] - self.lows[j])

                ema_val = calculate_average(tp_window)
                atr_val = calculate_average(hl_diff)
                upper_band = ema_val + (2 * atr_val)

                ema_list.append(ema_val)
                atr_list.append(atr_val)
                upper_list.append(upper_band)

        self.df['keltner_ema'] = ema_list
        self.df['keltner_atr'] = atr_list
        self.df['keltner_upper'] = upper_list

    # ----------------------------------------------------------------
    # 13) Supertrend
    # ----------------------------------------------------------------
    def supertrend(self, period=10, multiplier=3):
        """
        Snippet logic:
          tr = [high - low for j in range(i-period, i)]
          atr = average(tr)
          basic_upper = (max(high[i-period:i]) + min(low[i-period:i]))/2 + (multiplier*atr)
        We'll store 'supertrend_basic_upper'.
        """
        st_upper_list = []
        for i in range(len(self.closes)):
            if i < period:
                st_upper_list.append(np.nan)
            else:
                tr_list = []
                for j in range(i - period, i):
                    tr_list.append(self.highs[j] - self.lows[j])
                atr_val = calculate_average(tr_list)
                highest = np.max(self.highs[i - period : i])
                lowest = np.min(self.lows[i - period : i])
                basic_upper = (highest + lowest) / 2 + (multiplier * atr_val)
                st_upper_list.append(basic_upper)

        self.df['supertrend_basic_upper'] = st_upper_list

    # ----------------------------------------------------------------
    # 14) Fibonacci
    # ----------------------------------------------------------------
    def fibonacci(self, period=20):
        """
        Snippet logic:
          high = max(self.highs[i-period:i])
          low = min(self.lows[i-period:i])
          fib_618 = low + (high - low)*0.618
        We'll store fib_618 in 'fib_618_value'.
        """
        fib_list = []
        for i in range(len(self.closes)):
            if i < period:
                fib_list.append(np.nan)
            else:
                highest = np.max(self.highs[i - period : i])
                lowest = np.min(self.lows[i - period : i])
                fib_618 = lowest + (highest - lowest)*0.618
                fib_list.append(fib_618)

        self.df['fib_618_value'] = fib_list

    # ----------------------------------------------------------------
    # 15) Price Action
    # ----------------------------------------------------------------
    def price_action(self):
        """
        Snippet logic:
          bullish_candle = closes[i] > opens[i].
        We'll store a boolean for each row: True/False if bullish.
        """
        bullish_list = (self.df['Close'] > self.df['Open']).tolist()
        self.df['priceaction_bullish'] = bullish_list

    # ----------------------------------------------------------------
    # 16) Volume Spike
    # ----------------------------------------------------------------
    def volume_spike(self, period=20):
        """
        Snippet logic:
          avg_volume (last period) = average(self.volumes[i-period : i-1])
          If self.volumes[i] > (2 * avg_volume) => spike.
        We'll store 'vol_spike_avg' and a boolean 'vol_spike'.
        """
        vol_avg_list = []
        vol_spike_list = []
        for i in range(len(self.volumes)):
            if i < period:
                vol_avg_list.append(np.nan)
                vol_spike_list.append(False)
            else:
                avg_vol = calculate_average(self.volumes[i - period : i])
                vol_avg_list.append(avg_vol)
                if self.volumes[i] > 2 * avg_vol:
                    vol_spike_list.append(True)
                else:
                    vol_spike_list.append(False)

        self.df['vol_spike_avg'] = vol_avg_list
        self.df['vol_spike'] = vol_spike_list

    # ----------------------------------------------------------------
    # 17) Momentum
    # ----------------------------------------------------------------
    def momentum(self, period=10):
        """
        Snippet logic:
          momentum = closes[i] - closes[i-period].
        We'll store 'momentum_value'.
        """
        mom_list = []
        for i in range(len(self.closes)):
            if i < period:
                mom_list.append(np.nan)
            else:
                mom_val = self.closes[i] - self.closes[i - period]
                mom_list.append(mom_val)

        self.df['momentum_value'] = mom_list

    # ----------------------------------------------------------------
    # 18) Williams %R
    # ----------------------------------------------------------------
    def williams_r(self, period=14):
        """
        Snippet logic:
          highest_high = max(self.highs[i-period:i])
          lowest_low = min(self.lows[i-period:i])
          wr = ((highest_high - close[i]) / (highest_high - lowest_low)) * -100
        We'll store 'williams_r_value'.
        """
        wr_list = []
        for i in range(len(self.closes)):
            if i < period:
                wr_list.append(np.nan)
            else:
                hh = np.max(self.highs[i - period : i])
                ll = np.min(self.lows[i - period : i])
                denom = (hh - ll)
                if denom == 0:
                    wr = np.nan
                else:
                    wr = ((hh - self.closes[i]) / denom) * -100
                wr_list.append(wr)

        self.df['williams_r_value'] = wr_list

    # ----------------------------------------------------------------
    # 19) CMF
    # ----------------------------------------------------------------
    def cmf(self, period=20):
        """
        Snippet logic:
          mfv[j] = (((close - low) - (high - close)) / (high - low)) * volume[j]
          cmf = sum(mfv) / sum(volumes) for the last 'period' bars.
        We'll store 'cmf_value'.
        """
        cmf_list = []
        for i in range(len(self.closes)):
            if i < period:
                cmf_list.append(np.nan)
            else:
                mfv_vals = []
                vol_vals = []
                for j in range(i - period, i):
                    high_low = self.highs[j] - self.lows[j]
                    if high_low == 0:
                        mfv = 0
                    else:
                        mfv = (((self.closes[j] - self.lows[j]) - (self.highs[j] - self.closes[j])) 
                               / high_low) * self.volumes[j]
                    mfv_vals.append(mfv)
                    vol_vals.append(self.volumes[j])
                total_vol = np.sum(vol_vals)
                if total_vol == 0:
                    cmf_val = np.nan
                else:
                    cmf_val = np.sum(mfv_vals) / total_vol
                cmf_list.append(cmf_val)

        self.df['cmf_value'] = cmf_list

    # ----------------------------------------------------------------
    # 20) OBV
    # ----------------------------------------------------------------
    def obv(self):
        """
        Snippet logic:
          obv starts at 0, then for each j from 1..i:
            if close[j] > close[j-1]: obv += volume[j]
            elif close[j] < close[j-1]: obv -= volume[j]
        We'll store 'obv_value' for each row.
        """
        obv_values = []
        for i in range(len(self.closes)):
            if i == 0:
                obv_values.append(0)
            else:
                # compute OBV up to i
                obv_val = 0
                for j in range(1, i+1):
                    if self.closes[j] > self.closes[j-1]:
                        obv_val += self.volumes[j]
                    elif self.closes[j] < self.closes[j-1]:
                        obv_val -= self.volumes[j]
                obv_values.append(obv_val)
        self.df['obv_value'] = obv_values

    # ----------------------------------------------------------------
    # Method: run_all_indicators
    #   Dynamically calls every indicator method so you don't have to 
    #   manually call them one by one.
    # ----------------------------------------------------------------
    def run_all_indicators(self):
        """
        Dynamically finds and calls all indicator methods (those that 
        don't start with underscore or appear in the skip set).
        """
        skip_methods = {
            'run_all_indicators', 'clean_data', '__init__',
            '__module__', '__doc__', '__dict__', '__weakref__'
        }
        for attr_name in dir(self):
            if attr_name.startswith('_'):
                continue
            if attr_name in skip_methods:
                continue

            attr = getattr(self, attr_name)
            if callable(attr):
                print(f"Running: {attr_name}")
                attr()  # call the indicator method
        return self

    def get_new_indicator_columns(self):
        """
        Returns a list of columns that were added after the class 
        was initialized—i.e. newly created indicator columns.
        """
        current_columns = set(self.df.columns)
        # The newly added columns are the difference
        new_columns = current_columns - self.original_columns
        return list(new_columns)
    
    # ----------------------------------------------------------------
    # Method: clean_data
    #   Example cleanup: drop rows with NaN in newly created columns.
    #   Also, optionally remove rows with zeros if needed.
    # ----------------------------------------------------------------
    def clean_data(self):
        """
        Drop rows that contain NaN in any newly created indicator columns.
        """
        # Original columns in your dataset:
        original_cols = {'open', 'high', 'low', 'close', 'volume'}
        new_cols = [c for c in self.df.columns if c not in original_cols]

        # Drop rows where any of these new columns has NaN:
        self.df.dropna(subset=new_cols, inplace=True)

        # Optionally drop rows with zeros in these columns:
        # mask = (self.df[new_cols] == 0).any(axis=1)
        # self.df = self.df[~mask]

        return self.df

In [15]:
# ind_sys = StockIndicators(data)

In [19]:
# ind_sys = StockIndicators(data)
# df = ind_sys.run_all_indicators()
# df = ind_sys.clean_data()

In [20]:
# df = ind_sys.clean_data()

In [30]:
# Helper functions (unchanged)
def calculate_average(data):
    return sum(data) / len(data) if data else 0

def calculate_standard_deviation(data):
    if not data:
        return 0
    mean = calculate_average(data)
    variance = sum((x - mean) ** 2 for x in data) / len(data)
    return math.sqrt(variance)

def calculate_stochastic_rsi(prices, period=14):
    if len(prices) < period:
        return 0, 0
    
    rsi_values = []
    for i in range(len(prices) - 1):
        change = prices[i + 1] - prices[i]
        gain = change if change > 0 else 0
        loss = -change if change < 0 else 0
        rsi_values.append((gain, loss))
    
    if len(rsi_values) < period:
        return 0, 0
        
    gains = [x[0] for x in rsi_values[-period:]]
    losses = [x[1] for x in rsi_values[-period:]]
    avg_gain = calculate_average(gains)
    avg_loss = calculate_average(losses)
    
    rs = avg_gain / avg_loss if avg_loss != 0 else 0
    rsi = 100 - (100 / (1 + rs)) if rs != 0 else 0
    
    current_rsi = rsi
    rsi_high = max(rsi_values[-period:], key=lambda x: x[0])[0]
    rsi_low = min(rsi_values[-period:], key=lambda x: x[1])[1]
    
    k = ((current_rsi - rsi_low) / (rsi_high - rsi_low)) * 100 if rsi_high != rsi_low else 0
    d = calculate_average([k] * 3)
    return k, d

def calculate_metrics(positions, prices, initial_balance, transaction_cost=0.01):
    """Calculate trading metrics with transaction costs"""
    equity_curve = [initial_balance]
    profits = []
    losses = []
    current_equity = initial_balance
    peak = initial_balance
    max_drawdown = 0
    
    for i, (entry_price, exit_price, shares) in enumerate(positions):
        if entry_price and exit_price:
            pl = (exit_price - entry_price) * shares - (2 * transaction_cost * shares)
            current_equity += pl
            equity_curve.append(current_equity)
            peak = max(peak, current_equity)
            drawdown = peak - current_equity
            max_drawdown = max(max_drawdown, drawdown)
            
            if pl > 0:
                profits.append((i, pl))
            elif pl < 0:
                losses.append((i, pl))
    
    total_pl = current_equity - initial_balance
    return max_drawdown, profits, losses, total_pl, equity_curve

## Stratigies

In [36]:
class TradingStrategies:
    def __init__(self, df, initial_balance=5000):
        self.opens = df['open'].tolist()
        self.highs = df['high'].tolist()
        self.lows = df['low'].tolist()
        self.closes = df['close'].tolist()
        self.volumes = df['volume'].tolist()
        self.dates = df.index.tolist()
        self.initial_balance = initial_balance
        self.transaction_cost = 0.01  # $0.01 per share per transaction

    def run_strategy(self, signal_func, strategy_name):
        """Execute strategy with $0.01/share transaction costs"""
        positions = []
        entry_price = None
        available_balance = self.initial_balance
        trade_history = []
        equity_curve = [self.initial_balance]
        
        for i in range(len(self.closes)):
            signal = signal_func(i)
            current_date = self.dates[i]
            
            # Trade entry logic
            if signal and not entry_price and i < len(self.closes) - 1:
                entry_date = self.dates[i+1]
                entry_price = self.opens[i+1]
                max_shares = int(available_balance / (entry_price + self.transaction_cost))
                
                if max_shares > 0:
                    # Deduct entry cost
                    entry_cost = max_shares * self.transaction_cost
                    available_balance -= entry_cost
                    
                    trade = {
                        'entry_date': entry_date,
                        'entry_price': entry_price,
                        'exit_date': None,
                        'exit_price': None,
                        'shares': max_shares,
                        'entry_fee': entry_cost,
                        'exit_fee': 0
                    }
                    trade_history.append(trade)

            # Trade exit logic        
            elif entry_price and not signal:
                exit_date = current_date
                exit_price = self.closes[i]
                trade = trade_history[-1]
                
                # Calculate exit cost
                exit_cost = trade['shares'] * self.transaction_cost
                available_balance -= exit_cost
                
                # Update trade record
                trade['exit_date'] = exit_date
                trade['exit_price'] = exit_price
                trade['exit_fee'] = exit_cost
                
                # Calculate P/L
                pl = (exit_price - entry_price) * trade['shares']
                pl -= (trade['entry_fee'] + trade['exit_fee'])
                equity_curve.append(equity_curve[-1] + pl)
                
                entry_price = None

        # Close any open position
        if entry_price and trade_history and trade_history[-1]['exit_date'] is None:
            trade = trade_history[-1]
            exit_cost = trade['shares'] * self.transaction_cost
            available_balance -= exit_cost
            
            trade['exit_date'] = self.dates[-1]
            trade['exit_price'] = self.closes[-1]
            trade['exit_fee'] = exit_cost
            
            pl = (trade['exit_price'] - trade['entry_price']) * trade['shares']
            pl -= (trade['entry_fee'] + trade['exit_fee'])
            equity_curve.append(equity_curve[-1] + pl)

        # Calculate metrics
        metrics = self._calculate_metrics(trade_history, equity_curve)
        
        # Create result DataFrame
        result_df = pd.DataFrame({
            'date': self.dates,
            f'{strategy_name}_equity': equity_curve[:len(self.dates)],
            f'{strategy_name}_close': self.closes
        })
        
        return result_df, trade_history, metrics

    def _calculate_metrics(self, trades, equity_curve):
        """Calculate performance metrics with fee tracking"""
        metrics = {
            'total_pl': 0,
            'total_fees': 0,
            'max_drawdown': 0,
            'profits': [],
            'losses': []
        }

        if trades:
            # Calculate returns series
            returns = pd.Series(equity_curve).pct_change().dropna()
            
            # Maximum drawdown calculation
            cumulative = returns.cumsum()
            metrics['max_drawdown'] = (cumulative.cummax() - cumulative).max()

            # Trade-level metrics
            for trade in trades:
                if trade['exit_price'] is None:
                    continue
                
                # Calculate P/L after fees
                pl = ((trade['exit_price'] - trade['entry_price']) * trade['shares'] 
                      - trade['entry_fee'] - trade['exit_fee'])
                
                # Track fees
                metrics['total_fees'] += trade['entry_fee'] + trade['exit_fee']
                
                if pl > 0:
                    metrics['profits'].append(pl)
                else:
                    metrics['losses'].append(pl)

            metrics['total_pl'] = sum(metrics['profits']) + sum(metrics['losses'])

        return metrics

    def _generate_plots(self, results):
        """Generate complete visualizations for all strategies"""
        for strategy_name, data in results.items():
            self._plot_strategy_performance(strategy_name, data)

    def _plot_strategy_performance(self, strategy_name, data):
        """Create detailed performance visualization"""
        plt.figure(figsize=(18, 12))
        df = data['data']
        trades = data['trades']
        metrics = data['metrics']
        
        # Main Price Chart with Trades
        ax1 = plt.subplot2grid((4, 1), (0, 0), rowspan=2)
        ax1.plot(df['date'], df[f'{strategy_name}_close'], label='Price', color='navy')
        
        # Plot entry/exit markers
        entry_dates = [t['entry_date'] for t in trades if t['entry_date'] is not None]
        entry_prices = [t['entry_price'] for t in trades if t['entry_date'] is not None]
        exit_dates = [t['exit_date'] for t in trades if t['exit_date'] is not None]
        exit_prices = [t['exit_price'] for t in trades if t['exit_date'] is not None]
        
        ax1.scatter(entry_dates, entry_prices, marker='^', color='green', s=100, 
                   label='Entry', zorder=3)
        ax1.scatter(exit_dates, exit_prices, marker='v', color='red', s=100, 
                   label='Exit', zorder=3)
        ax1.set_title(f'{strategy_name} - Price with Trade Signals')
        ax1.legend()
        
        # Equity Curve
        ax2 = plt.subplot2grid((4, 1), (2, 0), sharex=ax1)
        ax2.plot(df['date'], df[f'{strategy_name}_equity'], 
                label='Equity', color='darkgreen')
        ax2.fill_between(df['date'], df[f'{strategy_name}_equity'], 
                        color='lightgreen', alpha=0.3)
        ax2.set_title('Equity Curve')
        
        # Drawdown Curve
        equity_series = pd.Series(df[f'{strategy_name}_equity'])
        cumulative_max = equity_series.cummax()
        drawdown = (cumulative_max - equity_series) / cumulative_max
        ax3 = plt.subplot2grid((4, 1), (3, 0), sharex=ax1)
        ax3.fill_between(df['date'], drawdown, color='orangered', alpha=0.3)
        ax3.set_title('Drawdown')
        
        # Metrics Table
        plt.subplot2grid((4, 1), (0, 0), rowspan=4, colspan=1)
        plt.axis('off')
        metric_text = (
            f"Total P/L: ${metrics['total_pl']:.2f}\n"
            f"Total Fees: ${metrics['total_fees']:.2f}\n"
            f"Max Drawdown: {metrics['max_drawdown']:.2%}\n"
            f"Profitable Trades: {len(metrics['profits'])}\n"
            f"Losing Trades: {len(metrics['losses'])}"
        )
        plt.text(0.05, 0.5, metric_text, fontsize=12, 
                bbox=dict(facecolor='white', alpha=0.8))

        plt.suptitle(f"{strategy_name} Performance Analysis\n"
                    f"Transaction Cost: $0.01 per share", y=1.02)
        plt.tight_layout()
        plt.savefig(f'{strategy_name}_performance.png', bbox_inches='tight')
        plt.close()

    def gaussian_channel(self, period=20):
        def signal(i):
            if i < period:
                return False
            mean = calculate_average(self.closes[i-period:i])
            std = calculate_standard_deviation(self.closes[i-period:i])
            upper_band = mean + (2 * std)
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return self.closes[i] > upper_band and k > d
        return self.run_strategy(signal, 'gaussian')

    def ma_crossover(self, fast_period=10, slow_period=20):
        def signal(i):
            if i < slow_period:
                return False
            fast_ma = calculate_average(self.closes[i-fast_period:i])
            slow_ma = calculate_average(self.closes[i-slow_period:i])
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            k_prev = calculate_stochastic_rsi(self.closes[:i])[0]
            return fast_ma > slow_ma and k > k_prev
        return self.run_strategy(signal, 'ma')

    def bollinger_bands(self, period=20):
        def signal(i):
            if i < period:
                return False
            mean = calculate_average(self.closes[i-period:i])
            std = calculate_standard_deviation(self.closes[i-period:i])
            upper_band = mean + (2 * std)
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return self.closes[i] > upper_band and k > 20
        return self.run_strategy(signal, 'bollinger')

    def parabolic_sar(self):
        def signal(i):
            if i < 2:
                return False
            sar = min(self.lows[:i])
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            k_prev = calculate_stochastic_rsi(self.closes[:i])[0]
            return self.closes[i] > sar and k > k_prev
        return self.run_strategy(signal, 'psar')

    def ichimoku_cloud(self, period=26):
        def signal(i):
            if i < period:
                return False
            tenkan = (max(self.highs[i-9:i]) + min(self.lows[i-9:i])) / 2
            kijun = (max(self.highs[i-period:i]) + min(self.lows[i-period:i])) / 2
            senkou_a = (tenkan + kijun) / 2
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return self.closes[i] > senkou_a and k > 50
        return self.run_strategy(signal, 'ichimoku')

    def macd(self, fast=12, slow=26, signal=9):
        def signal(i):
            if i < slow:
                return False
            ema_fast = calculate_average(self.closes[i-fast:i])
            ema_slow = calculate_average(self.closes[i-slow:i])
            macd_line = ema_fast - ema_slow
            signal_line = calculate_average(self.closes[i-signal:i])
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return macd_line > signal_line and k > d
        return self.run_strategy(signal, 'macd')

    def rsi(self, period=14):
        def signal(i):
            if i < period:
                return False
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            k_prev = calculate_stochastic_rsi(self.closes[:i])[0]
            rsi_val = calculate_stochastic_rsi(self.closes[:i+1])[0]
            return rsi_val < 30 and k > k_prev
        return self.run_strategy(signal, 'rsi')

    def adx(self, period=14):
        def signal(i):
            if i < period:
                return False
            up_moves = [self.highs[j] - self.highs[j-1] for j in range(i-period+1, i+1)]
            down_moves = [self.lows[j-1] - self.lows[j] for j in range(i-period+1, i+1)]
            adx = calculate_average([abs(u - d) for u, d in zip(up_moves, down_moves)])
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return adx > 25 and k > d
        return self.run_strategy(signal)

    def atr(self, period=14):
        def signal(i):
            if i < period:
                return False
            tr = [max(self.highs[j] - self.lows[j], 
                     abs(self.highs[j] - self.closes[j-1]),
                     abs(self.lows[j] - self.closes[j-1])) 
                  for j in range(i-period+1, i+1)]
            atr = calculate_average(tr)
            upper_band = self.closes[i] + atr
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            k_prev = calculate_stochastic_rsi(self.closes[:i])[0]
            return self.closes[i] > upper_band and k > k_prev
        return self.run_strategy(signal)

    def vwma(self, period=20):
        def signal(i):
            if i < period:
                return False
            price_vol = sum(p * v for p, v in zip(self.closes[i-period:i], self.volumes[i-period:i]))
            vol_sum = sum(self.volumes[i-period:i])
            vwma = price_vol / vol_sum if vol_sum != 0 else 0
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return self.closes[i] > vwma and k > d
        return self.run_strategy(signal)

    def donchian_channel(self, period=20):
        def signal(i):
            if i < period:
                return False
            upper_channel = max(self.highs[i-period:i])
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return self.closes[i] > upper_channel and k > 20
        return self.run_strategy(signal)

    def keltner_channel(self, period=20):
        def signal(i):
            if i < period:
                return False
            typical_price = [(h + l + c) / 3 for h, l, c in 
                           zip(self.highs[i-period:i], self.lows[i-period:i], self.closes[i-period:i])]
            ema = calculate_average(typical_price)
            atr = calculate_average([max(h - l) for h, l in zip(self.highs[i-period:i], self.lows[i-period:i])])
            upper_band = ema + (2 * atr)
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return self.closes[i] > upper_band and k > d
        return self.run_strategy(signal)

    def supertrend(self, period=10, multiplier=3):
        def signal(i):
            if i < period:
                return False
            tr = [max(self.highs[j] - self.lows[j]) for j in range(i-period, i)]
            atr = calculate_average(tr)
            basic_upper = (max(self.highs[i-period:i]) + min(self.lows[i-period:i])) / 2 + (multiplier * atr)
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return self.closes[i] > basic_upper and k > 20
        return self.run_strategy(signal)

    def fibonacci(self, period=20):
        def signal(i):
            if i < period:
                return False
            high = max(self.highs[i-period:i])
            low = min(self.lows[i-period:i])
            fib_618 = low + (high - low) * 0.618
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return abs(self.closes[i] - fib_618) < (high - low) * 0.05 and k > d
        return self.run_strategy(signal)

    def price_action(self):
        def signal(i):
            if i < 1:
                return False
            bullish_candle = self.closes[i] > self.opens[i]
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            k_prev = calculate_stochastic_rsi(self.closes[:i])[0]
            return bullish_candle and k > k_prev
        return self.run_strategy(signal)

    def volume_spike(self, period=20):
        def signal(i):
            if i < period:
                return False
            avg_volume = calculate_average(self.volumes[i-period:i-1])
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return self.volumes[i] > (2 * avg_volume) and k > d
        return self.run_strategy(signal)

    def momentum(self, period=10):
        def signal(i):
            if i < period:
                return False
            momentum = self.closes[i] - self.closes[i-period]
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            k_prev = calculate_stochastic_rsi(self.closes[:i])[0]
            return momentum > 0 and k > k_prev
        return self.run_strategy(signal)

    def williams_r(self, period=14):
        def signal(i):
            if i < period:
                return False
            highest_high = max(self.highs[i-period:i])
            lowest_low = min(self.lows[i-period:i])
            wr = ((highest_high - self.closes[i]) / (highest_high - lowest_low)) * -100
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            k_prev = calculate_stochastic_rsi(self.closes[:i])[0]
            return wr < -80 and k > k_prev
        return self.run_strategy(signal)

    def cmf(self, period=20):
        def signal(i):
            if i < period:
                return False
            mfv = [((self.closes[j] - self.lows[j]) - (self.highs[j] - self.closes[j])) / 
                   (self.highs[j] - self.lows[j]) * self.volumes[j] 
                   if (self.highs[j] - self.lows[j]) != 0 else 0 
                   for j in range(i-period, i)]
            cmf = sum(mfv) / sum(self.volumes[i-period:i]) if sum(self.volumes[i-period:i]) != 0 else 0
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            return cmf > 0 and k > d
        return self.run_strategy(signal)

    def obv(self):
        def signal(i):
            if i < 1:
                return False
            obv = 0
            for j in range(1, i+1):
                if self.closes[j] > self.closes[j-1]:
                    obv += self.volumes[j]
                elif self.closes[j] < self.closes[j-1]:
                    obv -= self.volumes[j]
            k, d = calculate_stochastic_rsi(self.closes[:i+1])
            k_prev = calculate_stochastic_rsi(self.closes[:i])[0]
            return obv > 0 and k > k_prev
        return self.run_strategy(signal)

In [ ]:
# # Example usage
# Initialize strategies
strategies = TradingStrategies(data)

# Run MACD strategy
result = strategies.gaussian_channel()

In [ ]:
# print(result[['date', 'equity', 'max_drawdown', 'total_pl']].head())
# print("\nSample profits/losses:")
# print(result.filter(like='profit').sum())
# print(result.filter(like='loss').sum())